# 3D Gaussian Splatting no Colab
Notebook baseado no repositório oficial: https://github.com/graphdeco-inria/gaussian-splatting

**Antes de rodar:** vá em `Ambiente de execução > Alterar tipo de ambiente de execução` e selecione uma GPU (T4 no free, A100/V100 se tiver Colab Pro).

## 1. Checar GPU disponível

In [ ]:
!nvidia-smi

Mon Jul  6 17:16:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Montar o Google Drive (para salvar checkpoints e resultados)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/gaussian_splatting_output', exist_ok=True)

Mounted at /content/drive


## 3. Clonar o repositório (com submódulos)

In [ ]:
!git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive
%cd gaussian-splatting

Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 1053, done.
remote: Total 1053 (delta 0), reused 0 (delta 0), pack-reused 1053 (from 1)
Receiving objects: 100% (1053/1053), 78.71 MiB | 31.62 MiB/s, done.
Resolving deltas: 100% (595/595), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compressing objects: 100% (17

## 4. Instalar dependências
No Colab é mais simples instalar via pip direto (em vez do `environment.yml`, que é pensado para conda local).

In [ ]:
!pip install plyfile tqdm opencv-python joblib

# Checar versão do PyTorch/CUDA já instalada no Colab
import torch
print('Torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU disponível:', torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.5 MB/s eta 0:00:00
Torch: 2.11.0+cu128 | CUDA: 12.8 | GPU disponível: True


## 5. Compilar os submódulos CUDA (rasterizador + simple-knn)
Essa etapa compila extensões C++/CUDA, pode levar alguns minutos. Se der erro aqui, normalmente é incompatibilidade entre a versão do CUDA do Colab e a esperada pelos submódulos — ver nota no final do notebook.

In [ ]:
!pip install submodules/diff-gaussian-rasterization
!pip install submodules/simple-knn

Processing ./submodules/diff-gaussian-rasterization
  Preparing metadata (setup.py) ... done
  Created wheel for diff_gaussian_rasterization: filename=diff_gaussian_rasterization-0.0.0-cp312-cp312-linux_x86_64.whl size=3816622 sha256=6e83f2ad9c6a726fb3061dd1375abdda4f659d31a26daf3a049bb1dc58f791c0
  Stored in directory: /root/.cache/pip/wheels/01/e0/e8/f40a1cd6a1d5760cbd3036081bdad5b36c41fc11c786d4a404
Successfully built diff_gaussian_rasterization
Processing ./submodules/simple-knn
  Preparing metadata (setup.py) ... done
  Created wheel for simple_knn: filename=simple_knn-0.0.0-cp312-cp312-linux_x86_64.whl size=3555484 sha256=c65c64b5a4e67593df5617a454e85229839225f6af7e836e4c6466dfff4e7d71
  Stored in directory: /root/.cache/pip/wheels/0a/f2/1b/255828ebad94ea248378281b7926639d83ce4f394f0052800d
Successfully built simple_knn


## 6. Baixar um dataset de exemplo (Tanks & Temples / Deep Blending)
Dataset oficial de exemplo (~650MB) disponibilizado pelos autores, já com as poses de câmera calculadas (não precisa rodar COLMAP).

In [ ]:
!wget https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip
!unzip -q tandt_db.zip
!ls tandt

--2026-07-06 17:22:45--  https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip
Resolving repo-sam.inria.fr (repo-sam.inria.fr)... 138.96.1.1
Connecting to repo-sam.inria.fr (repo-sam.inria.fr)|138.96.1.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 682628995 (651M) [application/zip]
Saving to: ‘tandt_db.zip’

tandt_db.zip        100%[===================>] 651.00M  14.3MB/s    in 60s     

2026-07-06 17:23:46 (10.8 MB/s) - ‘tandt_db.zip’ saved [682628995/682628995]

train  truck


## 6b. (Alternativa) Usar suas próprias fotos
Se preferir usar suas próprias imagens em vez do dataset de exemplo, suba as fotos para `/content/gaussian-splatting/data/input/` e rode o COLMAP para gerar as poses de câmera. Descomente as linhas abaixo.

In [ ]:
!apt-get install -y colmap
!colmap -h  # confirma que instalou

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
colmap is already the newest version (3.7-2).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
COLMAP 3.7 -- Structure-from-Motion and Multi-View Stereo
              (Commit Unknown on Unknown without CUDA)

Usage:
  colmap [command] [options]

Documentation:
  https://colmap.github.io/

Example usage:
  colmap help [ -h, --help ]
  colmap gui
  colmap gui -h [ --help ]
  colmap automatic_reconstructor -h [ --help ]
  colmap automatic_reconstructor --image_path IMAGES --workspace_path WORKSPACE
  colmap feature_extractor --image_path IMAGES --database_path DATABASE
  colmap exhaustive_matcher --database_path DATABASE
  colmap mapper --image_path IMAGES --database_path DATABASE --output_path MODEL
  ...

Available commands:
  help
  gui
  automatic_reconstructor
  bundle_adjuster
  color_extractor
  database_cleaner
  database_creator
  database_merger
  delaunay_mesher
  exh

In [ ]:
import shutil, os

SCENE_NAME = "meu_objeto"
scene_path = f"/content/gaussian-splatting/data"
input_path = f"{scene_path}/input"
os.makedirs(input_path, exist_ok=True)

# Define DRIVE_ROOT e subpastas antes de usá-las
DRIVE_ROOT = "/content/gaussian-splatting/data/input"  # Assume que DRIVE_IMAGES é a pasta raiz com as subpastas
subpastas = [d for d in os.listdir(DRIVE_ROOT) if os.path.isdir(os.path.join(DRIVE_ROOT, d))]

total = 0
for subpasta in subpastas:
    subpasta_path = os.path.join(DRIVE_ROOT, subpasta)
    if not os.path.isdir(subpasta_path):
        continue
    for fname in sorted(os.listdir(subpasta_path)):
        ext = os.path.splitext(fname)[1].lower()
        if ext not in [".jpg", ".jpeg", ".png"]:
            continue
        # prefixa com o nome da subpasta pra evitar colisão (imagem1.jpg em várias pastas)
        novo_nome = f"{subpasta}_{fname}"
        shutil.copy(
            os.path.join(subpasta_path, fname),
            os.path.join(input_path, novo_nome)
        )
        total += 1

print(f"{total} imagens copiadas de {len(subpastas)} subpastas para {input_path}")

0 imagens copiadas de 0 subpastas para /content/gaussian-splatting/data/input


In [ ]:
# 1. Extraia as features das imagens (pontos chave e descritores)
!colmap feature_extractor \
    --database_path {scene_path}/database.db \
    --image_path {scene_path}/input \
    --ImageReader.camera_model SIMPLE_PINHOLE \
    --SiftExtraction.use_gpu 0


Feature extraction

Processed file [1/33]
  Name:            IMG_0002.jpg
  Dimensions:      3000 x 4000
  Camera:          #1 - SIMPLE_PINHOLE
  Focal Length:    2628.57px (Prior)
  Features:        9770
Processed file [2/33]
  Name:            IMG_0001.jpg
  Dimensions:      3000 x 4000
  Camera:          #1 - SIMPLE_PINHOLE
  Focal Length:    2628.57px (Prior)
  Features:        10018
Processed file [3/33]
  Name:            IMG_0003.jpg
  Dimensions:      3000 x 4000
  Camera:          #1 - SIMPLE_PINHOLE
  Focal Length:    2628.57px (Prior)
  Features:        10178
Processed file [4/33]
  Name:            IMG_0004.jpg
  Dimensions:      3000 x 4000
  Camera:          #1 - SIMPLE_PINHOLE
  Focal Length:    2628.57px (Prior)
  Features:        13542
Processed file [5/33]
  Name:            IMG_0006.jpg
  Dimensions:      3000 x 4000
  Camera:          #1 - SIMPLE_PINHOLE
  Focal Length:    2628.57px (Prior)
  Features:        9583
Processed file [6/33]
  Name:            IMG_0005.j

In [ ]:
# 2. Encontre correspondências entre as features de todas as imagens
!colmap exhaustive_matcher \
    --database_path {scene_path}/database.db \
    --SiftMatching.use_gpu 0


Exhaustive feature matching

Matching block [1/1, 1/1] in 794.261s
Elapsed time: 13.238 [minutes]


In [ ]:
# 3. Reconstrua a cena 3D e estime as poses das câmeras
!mkdir -p {scene_path}/sparse
!colmap mapper \
    --database_path {scene_path}/database.db \
    --image_path {scene_path}/input \
    --output_path {scene_path}/sparse


Loading database

Loading cameras... 1 in 0.000s
Loading matches... 204 in 0.003s
Loading images... 33 in 0.022s (connected 33)
Building correspondence graph... in 0.031s (ignored 0)

Elapsed time: 0.001 [minutes]


Finding good initial image pair


Initializing with image pair #5 and #3


Global bundle adjustment

iter      cost      cost_change  |gradient|   |step|    tr_ratio  tr_radius  ls_iter  iter_time  total_time
   0  1.408156e+03    0.00e+00    6.64e+04   0.00e+00   0.00e+00  1.00e+04        0    8.78e-04    1.10e-02
   1  3.114707e+02    1.10e+03    6.16e+04   4.45e+01   9.60e-01  3.00e+04        1    6.08e-03    1.71e-02
   2  2.453238e+02    6.61e+01    1.61e+04   4.06e+00   8.28e-01  4.18e+04        1    1.17e-03    1.83e-02
   3  2.251949e+02    2.01e+01    8.93e+03   1.41e+01   8.84e-01  7.63e+04        1    1.20e-03    1.95e-02
   4  2.205149e+02    4.68e+00    4.01e+03   1.02e+01   9.08e-01  1.67e+05        1    1.14e-03    2.07e-02
   5  2.197408e+02    7.74e-01    

In [ ]:
import os
import shutil

# Define the paths based on previous cell's context
scene_path = '/content/gaussian-splatting/data'
input_dir = os.path.join(scene_path, 'input')
images_dir = os.path.join(scene_path, 'images')

# Create the target images directory if it doesn't exist
os.makedirs(images_dir, exist_ok=True)

# Move all image files from input_dir to images_dir
num_moved = 0
for fname in os.listdir(input_dir):
    if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
        src_path = os.path.join(input_dir, fname)
        dest_path = os.path.join(images_dir, fname)
        shutil.move(src_path, dest_path)
        num_moved += 1

print(f"Moved {num_moved} image files from {input_dir} to {images_dir}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/gaussian-splatting/data/input'

Now that the images are in the expected `images` subdirectory, you can retry running the `train.py` command. I will provide it again below.

## 7. Treinar
Usando a cena `truck` do Tanks & Temples como exemplo. Ajuste `-s` para o caminho do seu próprio dataset se estiver usando fotos próprias.

Se estiver na GPU T4 (16GB) e o treino der erro de memória (`CUDA out of memory`), reduza a densificação com `--densify_grad_threshold 0.0004` (padrão é 0.0002) ou use imagens em resolução menor.

In [ ]:
!python train.py -s /content/gaussian-splatting/data -m /content/drive/MyDrive/gaussian_splatting_output/truck --eval --iterations 30000

2026-07-06 19:45:28.119342: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing /content/drive/MyDrive/gaussian_splatting_output/truck
Output folder: /content/drive/MyDrive/gaussian_splatting_output/truck [06/07 19:45:32]
------------LLFF HOLD------------- [06/07 19:45:32]
Reading camera 33/33 [06/07 19:45:32]
Loading Training Cameras [06/07 19:45:32]
[ INFO ] Encountered quite large input images (>1.6K pixels width), rescaling to 1.6K.
 If this is not desired, please explicitly specify '--resolution/-r' as 1 [06/07 19:45:32]
Loading Test Cameras [06/07 19:45:41]
Number of points at initialisation :  12169 [06/07 19:45:42]
Training progress:  23% 7000/30000 [15:05<58:50,  6.51it/s, Loss=0.0694676, Depth Loss=0.0000000]
[ITER 7000] Evaluati

## 8. Renderizar e calcular métricas (comparação com baselines)
Gera as imagens renderizadas e depois calcula PSNR / SSIM / LPIPS.

In [ ]:
!python render.py -m /content/drive/MyDrive/gaussian_splatting_output/truck
!python metrics.py -m /content/drive/MyDrive/gaussian_splatting_output/truck

Looking for config file in /content/drive/MyDrive/gaussian_splatting_output/truck/cfg_args
Config file found: /content/drive/MyDrive/gaussian_splatting_output/truck/cfg_args
Rendering /content/drive/MyDrive/gaussian_splatting_output/truck
Loading trained model at iteration 30000 [06/07 21:11:54]
------------LLFF HOLD------------- [06/07 21:11:55]
Reading camera 33/33 [06/07 21:11:55]
Loading Training Cameras [06/07 21:11:55]
[ INFO ] Encountered quite large input images (>1.6K pixels width), rescaling to 1.6K.
 If this is not desired, please explicitly specify '--resolution/-r' as 1 [06/07 21:11:55]
Loading Test Cameras [06/07 21:12:03]
Rendering progress: 100% 28/28 [00:45<00:00,  1.61s/it]
Rendering progress: 100% 5/5 [00:08<00:00,  1.60s/it]

Scene: /content/drive/MyDrive/gaussian_splatting_output/truck
Method: ours_30000
Metric evaluation progress: 100% 32/32 [00:55<00:00,  1.72s/it]
  SSIM :    0.8309703
  PSNR :   23.3287544
  LPIPS:    0.2207660



## 9. (Opcional) Pipeline completo de avaliação vs baselines do paper
Isso reproduz treino + render + métricas para os datasets de referência (Mip-NeRF360, Tanks & Temples, Deep Blending) usados na comparação do paper. É pesado — nos autores leva ~7h numa A6000, no Colab free pode não ser viável rodar tudo de uma vez.

In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install lpips torchmetrics scikit-image
!pip install pillow numpy pandas matplotlib tqdm

Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 34.5 MB/s eta 0:00:00


In [ ]:
from eval_model import run_eval_from_scene

summary, df = run_eval_from_scene(
    scene=scene, gaussians=gaussians, pipeline=pipeline, bg=bg,
    render_fn=render, split="test", output_dir="eval_results",
)

ImportError: cannot import name 'run_eval_from_scene' from 'eval_model' (/content/eval_model.py)

In [ ]:
!cat full_eval.py

#
# Copyright (C) 2023, Inria
# GRAPHDECO research group, https://team.inria.fr/graphdeco
# All rights reserved.
#
# This software is free for non-commercial, research and evaluation use 
# under the terms of the LICENSE.md file.
#
# For inquiries contact  george.drettakis@inria.fr
#

import os
from argparse import ArgumentParser
import time

mipnerf360_outdoor_scenes = ["bicycle", "flowers", "garden", "stump", "treehill"]
mipnerf360_indoor_scenes = ["room", "counter", "kitchen", "bonsai"]
tanks_and_temples_scenes = ["truck", "train"]
deep_blending_scenes = ["drjohnson", "playroom"]

parser = ArgumentParser(description="Full evaluation script parameters")
parser.add_argument("--skip_training", action="store_true")
parser.add_argument("--skip_rendering", action="store_true")
parser.add_argument("--skip_metrics", action="store_true")
parser.add_argument("--output_path", default="./eval")
parser.add_argument("--use_depth", action="store_true")
parser.add_argument("--use_expcomp", action="